# Quetta valley, NISAR L2 GUNW: data processing

Processing notebook behind the dashboard at https://abdullahjan1020.github.io/quetta-nisar/ (NASA Space Apps Challenge 2026, "Dancing with the SARs").

**What it does:** finds NISAR L2 GUNW (Provisional) interferograms over the Quetta valley, crops nine consecutive pairs of one track, cleans them, chains them into cumulative **relative line-of-sight (LOS) change**, and exports the files the dashboard loads (`data/meta.json`, `data/cube.json`, `data/frames/*.png`).

**Read this first**
- The result is relative LOS change, **not** proven ground subsidence. The sign convention (which direction is toward the satellite) is **not confirmed**.
- Data are Provisional. About 2-3 cm of noise remains in areas that are probably stable.
- The ionospheric phase screen is subtracted. Troposphere and solid Earth tide corrections are **not** applied.
- Each pair is referenced to the median of its valid pixels (regional median), not to a validated stable point.

**How to run:** Google Colab, top to bottom. You need a free NASA Earthdata Login account. The password is typed into a hidden prompt; never write credentials into a cell. Before saving this notebook to GitHub use *Edit, Clear all outputs*.

## 1. Setup

In [ ]:
!pip install -q asf_search pyproj h5py
import os
import glob
import json
import shutil
import getpass
import numpy as np
import matplotlib.pyplot as plt
import h5py
import asf_search as asf
from PIL import Image
from pyproj import Transformer
from shapely.geometry import shape, box

# Study box (lon/lat, WGS84)
WEST, SOUTH, EAST, NORTH = 66.8, 30.0, 67.2, 30.35
BOX_WKT = 'POLYGON((66.8 30.0, 67.2 30.0, 67.2 30.35, 66.8 30.35, 66.8 30.0))'

# The track and frame that fully cover the box (see section 2)
TRACK, FRAME = 92, 73

# Layers we read from every GUNW file
GRID_BASE = 'science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/HH/'

# Phase (radians) to line-of-sight change (cm): lambda / (4 pi), lambda = c / center frequency (1.229 GHz)
CM_PER_RAD = (299792458 / 1.229e9) / (4 * np.pi) * 100
print('cm per radian:', round(CM_PER_RAD, 3))

## 2. Search for GUNW scenes and pick one frame

Layers used from each file: `unwrappedPhase`, `coherenceMagnitude`, `connectedComponents`, `ionospherePhaseScreen` (all 4302 x 4374 grids at 80 m, projection EPSG:32642) and their `xCoordinates` / `yCoordinates`.

In [ ]:
results = asf.search(
    granule_list=['NISAR_L2_PR_GUNW*'],
    intersectsWith=BOX_WKT,
    maxResults=250
)
print(len(results), 'GUNW scenes touch the box')

# How much of the box does each track/frame cover? (100 = fully)
quetta = box(WEST, SOUTH, EAST, NORTH)
coverage = []
for r in results:
    p = r.properties
    scene = shape(r.geometry)
    cover = scene.intersection(quetta).area / quetta.area * 100
    coverage.append((p['pathNumber'], p['frameNumber'], round(cover)))

for row in sorted(set(coverage)):
    print(row)

In [ ]:
# The consecutive pairs of the chosen track and frame, sorted by date
pairs = []
for r in results:
    p = r.properties
    if p['pathNumber'] == TRACK and p['frameNumber'] == FRAME:
        pairs.append((p['startTime'][:10], p['stopTime'][:10], p['url'], p['fileName']))

pairs = sorted(set(pairs))
print(len(pairs), 'pairs')
for item in pairs:
    print(item[0], '->', item[1])

## 3. Login and crop window

Each GUNW file is about 2.3 GB, so we download one, keep only the Quetta crop, and delete the file. The crop window is computed from the box corners converted to UTM zone 42N.

In [ ]:
username = input('Earthdata username: ')
password = getpass.getpass('Earthdata password: ')
session = asf.ASFSession().auth_with_creds(username, password)
del password

In [ ]:
to_utm = Transformer.from_crs('EPSG:4326', 'EPSG:32642', always_xy=True)

corners_lon = [WEST, EAST, EAST, WEST]
corners_lat = [SOUTH, SOUTH, NORTH, NORTH]
eastings = []
northings = []
for i in range(4):
    e, n = to_utm.transform(corners_lon[i], corners_lat[i])
    eastings.append(e)
    northings.append(n)

x_min = min(eastings)
x_max = max(eastings)
y_min = min(northings)
y_max = max(northings)
print('UTM box:', round(x_min), round(x_max), round(y_min), round(y_max))

## 4. Download, crop and save each pair (Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTDIR = '/content/drive/MyDrive/nisar_quetta'
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs('data', exist_ok=True)

for item in pairs:
    start = item[0]
    stop = item[1]
    url = item[2]
    fname = 'data/' + item[3]
    out = OUTDIR + '/' + start + '_' + stop + '.npz'

    if os.path.exists(out):
        print('skip', start, stop)
        continue

    if not os.path.exists(fname):
        resp = session.get(url, stream=True)
        if resp.status_code != 200:
            print('download failed', start, stop, resp.status_code)
            continue
        with open(fname, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)

    with h5py.File(fname, 'r') as f:
        xs = f[GRID_BASE + 'xCoordinates'][:]
        ys = f[GRID_BASE + 'yCoordinates'][:]
        c0 = int(np.argmin(np.abs(xs - x_min)))
        c1 = int(np.argmin(np.abs(xs - x_max)))
        r0 = int(np.argmin(np.abs(ys - y_max)))
        r1 = int(np.argmin(np.abs(ys - y_min)))
        ph = f[GRID_BASE + 'unwrappedPhase'][r0:r1, c0:c1]
        co = f[GRID_BASE + 'coherenceMagnitude'][r0:r1, c0:c1]
        cc = f[GRID_BASE + 'connectedComponents'][r0:r1, c0:c1]
        io = f[GRID_BASE + 'ionospherePhaseScreen'][r0:r1, c0:c1]

    np.savez_compressed(out, phase=ph, coh=co, cc=cc, iono=io, rows=[r0, r1], cols=[c0, c1])
    os.remove(fname)

    good = co > 0.5
    print(start, stop, 'coh', round(float(np.mean(co)), 2), 'good', round(float(np.mean(good)), 2),
          'iono std', round(float(np.std(io)), 2), 'shape', ph.shape)

In [ ]:
# Sanity check: all nine crops must use the same window of the same grid
npz_files = sorted(glob.glob(OUTDIR + '/????-??-??_????-??-??.npz'))
windows = set()
for path in npz_files:
    d = np.load(path)
    windows.add((int(d['rows'][0]), int(d['rows'][1]), int(d['cols'][0]), int(d['cols'][1])))

print(len(npz_files), 'pair files')
print('crop windows (should be exactly one):', windows)

## 5. Clean and chain the pairs

Per pair: mask pixels with coherence <= 0.5 or no connected component, subtract the ionospheric phase screen, subtract the median of the valid pixels, convert to cm. Then add the pairs up in date order. A pixel that is masked in any pair stays masked from that date on (NaN propagates), so the valid area shrinks over time.

In [ ]:
first = np.load(npz_files[0])
shape_crop = first['phase'].shape

dates = [os.path.basename(npz_files[0])[:10]]
cum_raw = [np.zeros(shape_crop)]    # without ionosphere correction (for comparison)
cum_iono = [np.zeros(shape_crop)]   # ionosphere subtracted (used for the dashboard)

for fpath in npz_files:
    stop_date = os.path.basename(fpath)[11:21]

    d = np.load(fpath)
    ph = d['phase'].astype('float64')
    co = d['coh']
    cc = d['cc']
    io = d['iono'].astype('float64')

    good = (co > 0.5) & (cc > 0)

    a = ph.copy()
    a[good == False] = np.nan
    a = a - np.nanmedian(a)

    b = ph - io
    b[good == False] = np.nan
    b = b - np.nanmedian(b)

    cum_raw.append(cum_raw[-1] + a * CM_PER_RAD)
    cum_iono.append(cum_iono[-1] + b * CM_PER_RAD)
    dates.append(stop_date)

for k in range(len(dates)):
    valid = 1 - np.mean(np.isnan(cum_iono[k]))
    print(dates[k], 'valid', round(float(valid), 2),
          'std raw', round(float(np.nanstd(cum_raw[k])), 2),
          'std iono-corr', round(float(np.nanstd(cum_iono[k])), 2))

np.savez_compressed(OUTDIR + '/cumulative.npz', dates=dates,
                    raw=np.array(cum_raw), iono=np.array(cum_iono))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
m1 = axes[0].imshow(cum_raw[-1], cmap='RdBu', vmin=-6, vmax=6)
plt.colorbar(m1, ax=axes[0], label='cm')
axes[0].set_title('cumulative, iono NOT removed')
m2 = axes[1].imshow(cum_iono[-1], cmap='RdBu', vmin=-6, vmax=6)
plt.colorbar(m2, ax=axes[1], label='cm')
axes[1].set_title('cumulative, iono removed')

s_raw = []
s_io = []
for k in range(len(dates)):
    s_raw.append(np.nanstd(cum_raw[k]))
    s_io.append(np.nanstd(cum_iono[k]))
axes[2].plot(dates, s_raw, label='raw')
axes[2].plot(dates, s_io, label='iono removed')
axes[2].set_ylabel('spatial std (cm)')
axes[2].tick_params(axis='x', rotation=60)
axes[2].legend()
plt.tight_layout()
plt.show()

## 6. Exploratory check of a candidate area

**Caveat:** the candidate patch below was picked by eye from the final cumulative map, so it is expected to end high. The two control boxes were also picked by hand and wander by about +-2-3 cm. This is a description of one area, **not evidence of ground deformation**.

Grid origin and spacing (from the GUNW `xCoordinates[0]`, `yCoordinates[0]`, 80 m spacing) are used to convert crop pixels to lon/lat.

In [ ]:
GRID_X0 = 252040.0     # easting of column 0 of the full GUNW grid (m)
GRID_Y0 = 3568280.0    # northing of row 0 of the full GUNW grid (m)
STEP = 80.0            # pixel size (m)

boxes = {
    'patch': (230, 290, 215, 250),   # rows from, rows to, cols from, cols to (crop pixels)
    'east': (100, 200, 380, 480),
    'west': (300, 400, 10, 100),
}

plt.figure(figsize=(9, 5))
for name in boxes:
    r0b = boxes[name][0]
    r1b = boxes[name][1]
    c0b = boxes[name][2]
    c1b = boxes[name][3]
    series = []
    for k in range(len(dates)):
        series.append(np.nanmean(cum_iono[k][r0b:r1b, c0b:c1b]))
    plt.plot(dates, series, marker='o', label=name)
    print(name, 'final value (cm):', round(float(series[-1]), 2))

plt.ylabel('cumulative LOS change (cm)')
plt.xticks(rotation=60)
plt.legend()
plt.grid(True)
plt.show()

to_lonlat = Transformer.from_crs('EPSG:32642', 'EPSG:4326', always_xy=True)
row0_full = int(first['rows'][0])
col0_full = int(first['cols'][0])
pr = (230 + 290) // 2
pc = (215 + 250) // 2
x = GRID_X0 + (col0_full + pc) * STEP
y = GRID_Y0 - (row0_full + pr) * STEP
lon, lat = to_lonlat.transform(x, y)
print('patch centre lon, lat:', round(lon, 4), round(lat, 4))

## 7. Export the dashboard files

Resamples the UTM crop (nearest neighbour) to a 160 x 160 lon/lat grid (row 0 = north, column 0 = west), writes one RGBA overlay per date, `cube.json` (values in cm, `null` = masked) and `meta.json`, then zips and downloads them. Colors run from -6 cm (red) through 0 (white) to +6 cm (blue); this is only a color scale, not a statement about up or down.

The overlay opacity grows with the size of the change: fully transparent below about 0.3 cm and fully strong from about 3 cm (rule fitted to match the frames that are deployed on the live site, accurate to about 1 out of 255). Transparent therefore means either masked or almost no change; `cube.json` has the exact values.

In [ ]:
rows_out = 160
cols_out = 160

H = cum_iono[0].shape[0]
Wd = cum_iono[0].shape[1]

# For every output pixel (north to south, west to east) find the crop row/col it comes from
to_utm_fwd = Transformer.from_crs('EPSG:4326', 'EPSG:32642', always_xy=True)
src_row = np.zeros((rows_out, cols_out), dtype=int)
src_col = np.zeros((rows_out, cols_out), dtype=int)
inside = np.zeros((rows_out, cols_out), dtype=bool)

for i in range(rows_out):
    lat = NORTH - (i + 0.5) * (NORTH - SOUTH) / rows_out
    for j in range(cols_out):
        lon = WEST + (j + 0.5) * (EAST - WEST) / cols_out
        x, y = to_utm_fwd.transform(lon, lat)
        rr = int(round((GRID_Y0 - y) / STEP)) - row0_full
        cc_ = int(round((x - GRID_X0) / STEP)) - col0_full
        if rr >= 0 and rr < H and cc_ >= 0 and cc_ < Wd:
            src_row[i, j] = rr
            src_col[i, j] = cc_
            inside[i, j] = True

print('pixels inside crop:', round(float(inside.mean()), 3))

out = []
for k in range(len(dates)):
    g = np.full((rows_out, cols_out), np.nan)
    for i in range(rows_out):
        for j in range(cols_out):
            if inside[i, j]:
                g[i, j] = cum_iono[k][src_row[i, j], src_col[i, j]]
    out.append(g)

# First date is zero by definition, but only where the last date is valid (same footprint)
mask_final = np.isnan(out[-1]) == False
out[0] = np.where(mask_final, 0.0, np.nan)

In [ ]:
VMIN = -6.0
VMAX = 6.0
LOW = np.array([178, 24, 43])      # #b2182b
MID = np.array([247, 247, 247])    # #f7f7f7
HIGH = np.array([33, 102, 172])    # #2166ac

def alpha_from_value(v):
    # 0 below ~0.3 cm, 220 from ~3 cm, linear in between
    t = (abs(v) - 0.3) / 2.7
    t = min(max(t, 0.0), 1.0)
    return int(round(t * 220))

def to_rgba(grid):
    h, w = grid.shape
    img = np.zeros((h, w, 4), dtype=np.uint8)
    for i in range(h):
        for j in range(w):
            v = grid[i, j]
            if np.isnan(v):
                continue
            t = (v - VMIN) / (VMAX - VMIN)
            t = min(max(t, 0.0), 1.0)
            if t < 0.5:
                col = LOW + (MID - LOW) * (t * 2)
            else:
                col = MID + (HIGH - MID) * ((t - 0.5) * 2)
            img[i, j, 0] = int(col[0])
            img[i, j, 1] = int(col[1])
            img[i, j, 2] = int(col[2])
            img[i, j, 3] = alpha_from_value(v)
    return img

root = OUTDIR + '/export'
os.makedirs(root + '/data/frames', exist_ok=True)

frame_paths = []
for k in range(len(dates)):
    Image.fromarray(to_rgba(out[k])).save(root + '/data/frames/' + dates[k] + '.png')
    frame_paths.append('data/frames/' + dates[k] + '.png')

vals = []
for k in range(len(dates)):
    flat = out[k].flatten()
    one = []
    for v in flat:
        if np.isnan(v):
            one.append(None)
        else:
            one.append(round(float(v), 2))
    vals.append(one)

with open(root + '/data/cube.json', 'w') as f:
    json.dump({'rows': rows_out, 'cols': cols_out, 'dates': dates, 'values': vals}, f)

meta = {
    'is_dummy': False,
    'title': 'Quetta valley: relative ground movement seen by NISAR',
    'units': 'cm',
    'quantity': 'Relative line-of-sight (LOS) change since 2026-06-20, referenced to the regional median of each pair',
    'bounds': {'west': WEST, 'south': SOUTH, 'east': EAST, 'north': NORTH},
    'dates': dates,
    'frames': frame_paths,
    'colormap': {'vmin': VMIN, 'vmax': VMAX, 'legend_stops': ['#b2182b', '#f7f7f7', '#2166ac']},
    'reference_point': None,
    'grid': {'rows': rows_out, 'cols': cols_out},
    'notes': [
        'NISAR L2 GUNW, Provisional quality. Track 92, Frame 73, descending, L-band HH.',
        '9 consecutive interferogram pairs chained (12-day, one 24-day gap pair) from 2026-06-20 to 2026-10-18.',
        'Pixels with coherence <= 0.5 or no connected component in any pair are masked.',
        'Ionospheric phase screen subtracted. Troposphere and solid Earth tide corrections NOT applied.',
        'Each pair referenced to the median of valid pixels in the Quetta box, not to a single stable point.',
        'LOS displacement: sign convention not yet confirmed. Noise level of 2-3 cm exists in apparently stable areas.'
    ]
}

with open(root + '/data/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

plt.figure(figsize=(6, 6))
plt.imshow(out[-1], cmap='RdBu', vmin=VMIN, vmax=VMAX)
plt.colorbar(label='cm')
plt.title('last date, lon/lat grid (north up)')
plt.show()

# Zip and download (Colab only)
from google.colab import files as colab_files
shutil.make_archive('/content/quetta_data', 'zip', root)
colab_files.download('/content/quetta_data.zip')

In [ ]:
# Check: mean of the valid 3x3 pixels around the candidate patch centre in the exported grid
r = 84
c = 76
patch = out[-1][r - 1:r + 2, c - 1:c + 2]
valid = patch[~np.isnan(patch)]

if len(valid) > 0:
    print('Valid pixels:', len(valid), ' mean LOS change:', round(float(valid.mean()), 2), 'cm')
else:
    print('No reliable data in this neighbourhood.')

## 8. Looking for a stable reference area (screening only)

A reference should be a place that does not move. An earlier version of this screen used only low variability between pairs (`temporal_std < 1 cm`) and high coherence, but that also selects pixels that change at a *steady* rate, including the candidate patch above. The version below also requires a mean pair change close to zero and leaves out the candidate area. It is a screening step; the result still needs to be judged on the map and against independent knowledge of the area before it is used as a reference.

In [ ]:
pair_changes = []
pair_coherence = []

for path in npz_files:
    d = np.load(path)

    ph = d['phase'].astype(float)
    io = d['iono'].astype(float)
    coh = d['coh']
    cc = d['cc']

    good = (coh > 0.5) & (cc > 0)

    change = ph - io
    change[good == False] = np.nan
    change = change - np.nanmedian(change)
    pair_changes.append(change * CM_PER_RAD)

    coh_clean = coh.astype(float).copy()
    coh_clean[good == False] = np.nan
    pair_coherence.append(coh_clean)

changes = np.array(pair_changes)
coherences = np.array(pair_coherence)

valid_count = np.sum(np.isfinite(changes), axis=0)
temporal_std = np.nanstd(changes, axis=0)
mean_change = np.nanmean(changes, axis=0)
mean_coh = np.nanmean(coherences, axis=0)

temporal_std[valid_count < 8] = np.nan

# Keep the candidate patch (plus a margin) out of the stable set
candidate = np.zeros(shape_crop, dtype=bool)
candidate[200:320, 185:280] = True

stable = (temporal_std < 1.0) & (mean_coh > 0.7) & (np.abs(mean_change) < 0.2) & (candidate == False)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
a = axes[0].imshow(mean_change, cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar(a, ax=axes[0], label='mean pair change (cm)')
axes[0].set_title('Mean change per pair')
b = axes[1].imshow(temporal_std, cmap='viridis', vmin=0, vmax=2)
plt.colorbar(b, ax=axes[1], label='Temporal std (cm)')
axes[1].set_title('Temporal variability')
c = axes[2].imshow(stable, cmap='gray')
axes[2].set_title('Possible stable pixels (screening only)')
plt.tight_layout()
plt.show()

print('Pixels with 8+ valid pairs:', int(np.sum(valid_count >= 8)))
print('Possible stable pixels:', int(np.sum(stable)))

## Known limitations

- Provisional data, not fully validated; about 2-3 cm of noise in apparently stable areas.
- No troposphere or solid Earth tide correction; regional-median reference only.
- One descending track: line-of-sight change only. Sign convention not confirmed.
- About 57% of pixels remain valid by the last date; four months is a short record.
- The candidate area was chosen by eye (selection bias); it needs independent validation.